# Evaluation and profiling of neighborhood clusters

This notebook examines the cluster assignments from the KMeans model,
profiles each segment by its mean feature values, creates heatmaps of
cluster characteristics, and provides interpretive labels for each
neighborhood segment.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, '..')
from src.data_loader import build_feature_matrix, FEATURE_COLUMNS
from src.model import (
    load_model, profile_clusters, fit_pca
)

pd.set_option('display.max_columns', 50)

## 1. Load model and data

In [ ]:
km = load_model('kmeans_model.joblib')
raw_df, scaled_df, scaler = build_feature_matrix()
X = scaled_df[FEATURE_COLUMNS].values

labels = km.labels_
n_clusters = len(set(labels))
print(f"Number of clusters: {n_clusters}")
print(f"Cluster sizes: {pd.Series(labels).value_counts().sort_index().to_dict()}")

## 2. Cluster profiles (mean feature values)

The profile table shows the average of each original-scale indicator
within each cluster, revealing what distinguishes the segments.

In [ ]:
profile = profile_clusters(raw_df, labels, FEATURE_COLUMNS)
print(profile.to_string(index=False))

## 3. Heatmap of cluster characteristics

A heatmap of z-scored cluster means makes cross-cluster comparisons
easy. Values above zero indicate a cluster is above average on that
dimension; below zero means below average.

In [ ]:
# Standardise the profile for the heatmap
profile_features = profile[FEATURE_COLUMNS]
profile_z = (profile_features - profile_features.mean()) / profile_features.std()
profile_z.index = [f'Cluster {c}' for c in profile['cluster']]

fig = px.imshow(
    profile_z,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Cluster profiles (z-scored means)',
    aspect='auto'
)
fig.update_layout(height=350, width=800)
fig.show()

## 4. Radar chart comparison

A radar chart provides another view of how clusters differ across all
indicators simultaneously. We use the z-scored means so features are
comparable.

In [ ]:
# Normalise to 0-1 range for radar chart readability
profile_norm = profile[FEATURE_COLUMNS].copy()
for col in FEATURE_COLUMNS:
    mn, mx = profile_norm[col].min(), profile_norm[col].max()
    if mx > mn:
        profile_norm[col] = (profile_norm[col] - mn) / (mx - mn)
    else:
        profile_norm[col] = 0.5

fig = go.Figure()
for i, row in profile_norm.iterrows():
    values = row[FEATURE_COLUMNS].tolist()
    values.append(values[0])  # close the polygon
    cats = FEATURE_COLUMNS + [FEATURE_COLUMNS[0]]
    fig.add_trace(go.Scatterpolar(
        r=values, theta=cats, fill='toself',
        name=f'Cluster {profile.iloc[i]["cluster"]}'
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Cluster radar chart (min-max normalised)',
    height=500
)
fig.show()

## 5. Communities per cluster

In [ ]:
raw_df_with_labels = raw_df.copy()
raw_df_with_labels['cluster'] = labels

for cl in sorted(raw_df_with_labels['cluster'].unique()):
    communities = raw_df_with_labels[raw_df_with_labels['cluster'] == cl]['community'].tolist()
    print(f"\nCluster {cl} ({len(communities)} communities):")
    print(', '.join(sorted(communities)[:15]))
    if len(communities) > 15:
        print(f'  ... and {len(communities) - 15} more')

## 6. PCA scatter with cluster labels

In [ ]:
pca, X_pca = fit_pca(X, n_components=2)

pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'cluster': labels.astype(str),
    'community': raw_df['community']
})

fig = px.scatter(
    pca_df, x='PC1', y='PC2', color='cluster',
    hover_data=['community'],
    title='Clusters visualised in PCA space',
    labels={'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
            'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})'},
    category_orders={'cluster': [str(i) for i in range(n_clusters)]}
)
fig.update_layout(height=500)
fig.show()

## 7. Feature distributions by cluster

In [ ]:
raw_df_with_labels['cluster_str'] = raw_df_with_labels['cluster'].astype(str)

# Show box plots for selected features
key_features = ['total_population', 'crime_rate', 'business_count', 'avg_building_cost']

fig = make_subplots(rows=2, cols=2, subplot_titles=key_features)
for i, feat in enumerate(key_features):
    r, c = divmod(i, 2)
    for cl in sorted(raw_df_with_labels['cluster'].unique()):
        subset = raw_df_with_labels[raw_df_with_labels['cluster'] == cl]
        fig.add_trace(
            go.Box(y=subset[feat], name=f'Cl {cl}', showlegend=(i == 0)),
            row=r + 1, col=c + 1
        )

fig.update_layout(height=600, title_text='Key feature distributions by cluster')
fig.show()

## 8. Segment naming and interpretation

Based on the cluster profiles, we assign descriptive names to each
segment. The exact labels depend on the data, but a typical naming
scheme follows this pattern.

In [ ]:
# Assign descriptive names based on observed profile characteristics
# These labels should be validated against the actual profile output above
segment_descriptions = {
    0: ('Urban core / high-activity', 
        'High population density, high business count, elevated crime rate, '
        'high permit activity. These are established inner-city communities '
        'with active commercial corridors.'),
    1: ('Suburban residential', 
        'Moderate population, low crime, few businesses, moderate housing costs. '
        'These communities are primarily residential with limited commercial '
        'activity.'),
    2: ('Developing / new communities', 
        'Lower population, high permit counts and building costs, low crime. '
        'These are newer neighborhoods experiencing active construction and '
        'growth.'),
}

for cluster_id, (name, description) in segment_descriptions.items():
    n_comm = (labels == cluster_id).sum()
    print(f"Cluster {cluster_id}: {name} ({n_comm} communities)")
    print(f"  {description}\n")

## 9. Summary

The neighborhood segmentation identified distinct community types within
Calgary based on census, crime, business, and housing indicators.

Key findings:

- Clusters separate clearly along population density, commercial
  activity, and development intensity axes.
- The heatmap and radar chart reveal that no single feature drives the
  segmentation; it is the combination of indicators that defines each
  segment.
- These segments can inform city planning, resource allocation, and
  targeted neighbourhood improvement strategies.

The segmentation model can be rerun periodically as new data becomes
available, allowing the city to track how neighborhoods evolve over time.